In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from imblearn.over_sampling import ADASYN
from category_encoders import TargetEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import nevergrad as ng
from collections import Counter

# --- 1. Load Data and Initial Preprocessing ---

df = pd.read_csv("./bank-additional/bank-additional/bank-additional-full.csv",delimiter=";")

# Drop columns as per original script
df.drop(['emp.var.rate','contact','housing','loan','previous','marital'], axis=1, inplace=True)

# Feature engineering and dropping columns
df['duration'] = (df['duration']/60).round()
df.drop(['day_of_week','default','pdays'], axis=1, inplace=True)

# Convert target variable
df['y'] = df['y'].map({'yes': 1, 'no': 0})

# Outlier handling for 'duration'
Q1_duration = df['duration'].quantile(0.25)
Q3_duration = df['duration'].quantile(0.75)
IQR_duration = Q3_duration - Q1_duration
lower_bound_duration = Q1_duration - 1.5 * IQR_duration
upper_bound_duration = Q3_duration + 1.5 * IQR_duration
df = df[(df['duration'] >= max(0, lower_bound_duration)) & 
        (df['duration'] <= upper_bound_duration)]

# Outlier handling for 'campaign'
Q1_campaign = df['campaign'].quantile(0.25)
Q3_campaign = df['campaign'].quantile(0.75)
IQR_campaign = Q3_campaign - Q1_campaign
lower_bound_campaign = Q1_campaign - 1.5 * IQR_campaign
upper_bound_campaign = Q3_campaign + 1.5 * IQR_campaign
df = df[(df['campaign'] >= max(1, lower_bound_campaign)) & 
        (df['campaign'] <= upper_bound_campaign)]

# Handle 'unknown' categories
df = df[df['education'] != 'unknown']
df = df[df['job'] != 'unknown']


# --- 2. Separate Features (X) and Target (y) ---

X = df.drop('y', axis=1)
y = df['y']


# --- 3. Train-Test Split (Crucial step before Target Encoding) ---

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print(f"Original X_train shape: {X_train.shape}")


# --- 4. Leakage-Free Target Encoding ---

# Define the columns to be target encoded
encoder_cols = ['education', 'job', 'month', 'poutcome']

# Initialize the encoder (TargetEncoder is now inside the cross-validation and testing context)
target_encoder = TargetEncoder(cols=encoder_cols, smoothing=1.0) 

# FIT the encoder ONLY on the training data (X_train, y_train)
X_train_encoded = target_encoder.fit_transform(X_train, y_train)

# TRANSFORM the test data using the FIT object (leakage-free)
X_test_encoded = target_encoder.transform(X_test) 

print(f"Encoded X_train shape: {X_train_encoded.shape}")


# --- 5. ADASYN Resampling (ONLY on Training Data) ---

# Prepare data for Naive Bayes (NB) and Decision Tree (DT) hyperparameter tuning
# Note: For CV during hyperparameter tuning (step 6 & 7), resampling is NOT applied to X_train/y_train

# Store the split data for later use in NB tuning
X_trainNB, X_testNB, y_trainNB, y_testNB = X_train_encoded.copy(), X_test_encoded.copy(), y_train.copy(), y_test.copy()

# Apply ADASYN ONLY to the training set for final model fitting
ismote = ADASYN(random_state=42, sampling_strategy = "auto", n_neighbors = 18)
X_train_res, y_train_res = ismote.fit_resample(X_train_encoded, y_train)

X_train_res_nb, y_train_res_nb = X_train_res, y_train_res # Store for Naive Bayes final fitting

print(f"\nResampled X_train shape: {X_train_res.shape}")
print(f"Class distribution after ADASYN: {Counter(y_train_res)}")

# --- 6. Decision Tree Hyperparameter Tuning (using Nevergrad) ---

def evaluate_decision_tree(criterion, max_depth, min_samples_split, min_samples_leaf):
    """Evaluates a Decision Tree model using cross-validation on unresampled X_train."""

    if max_depth == -1:
        max_depth = None

    clf = DecisionTreeClassifier(
        criterion=criterion,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        random_state=42
    )
    # CV is performed on the original, encoded, but UNRESAMPLED training set
    scores = cross_val_score(clf, X_train_encoded, y_train, cv=5, scoring='accuracy') 
    return -np.mean(scores) # Nevergrad minimizes this function

param_space = ng.p.Instrumentation(
    criterion=ng.p.Choice(['gini', 'entropy', 'log_loss']),
    max_depth=ng.p.Choice([-1] + list(range(1, 21))),
    min_samples_split=ng.p.Scalar(lower=2, upper=10).set_integer_casting(),
    min_samples_leaf=ng.p.Scalar(lower=1, upper=5).set_integer_casting(),
)

optimizer = ng.optimizers.OnePlusOne(parametrization=param_space, budget=50)
best_suggestion = optimizer.minimize(
    lambda *args, **kwargs: evaluate_decision_tree(*args, **kwargs)
)

best_kwargs = best_suggestion.kwargs
best_params_dt = {
    'criterion': best_kwargs['criterion'],
    'max_depth': None if best_kwargs['max_depth'] == -1 else int(best_kwargs['max_depth']),
    'min_samples_split': int(best_kwargs['min_samples_split']),
    'min_samples_leaf': int(best_kwargs['min_samples_leaf'])
}

# --- 7. Final Decision Tree Model Fitting (using Resampled Data) ---

clf_best = DecisionTreeClassifier(**best_params_dt, random_state=42) # Use 42 for consistency
clf_best.fit(X_train_res, y_train_res) # Fit on RESAMPLED data
y_pred_dt = clf_best.predict(X_test_encoded) # Predict on encoded test data

print("\n--- Decision Tree (Optimized) Results ---")
print(f"Best DT Params: {best_params_dt}")
print(f"Accuracy: {accuracy_score(y_test, y_pred_dt):.4f}")

feature_importances = pd.Series(clf_best.feature_importances_, index=X_train_encoded.columns)
print("\nTop 5 Feature Importances:")
print(feature_importances.sort_values(ascending=False).head())

# --- 8. Naive Bayes Hyperparameter Tuning (using Nevergrad) ---

def evaluate_nb(log10_var_smoothing):
    """Evaluates a GaussianNB model using cross-validation on unresampled X_train."""
    var_smoothing = 10.0 ** log10_var_smoothing
    clf_ = GaussianNB(var_smoothing=var_smoothing)
    # CV is performed on the original, encoded, but UNRESAMPLED training set
    scores = cross_val_score(clf_, X_trainNB, y_trainNB, cv=5, scoring='accuracy') 
    return -np.mean(scores)

param_space_nb = ng.p.Instrumentation(
    log10_var_smoothing=ng.p.Scalar(lower=-12, upper=-6)
)

optimizer_nb = ng.optimizers.OnePlusOne(parametrization=param_space_nb, budget=50)
best_suggestion_nb = optimizer_nb.minimize(lambda *a, **k: evaluate_nb(*a, **k))
best_log10_vs = float(best_suggestion_nb.kwargs['log10_var_smoothing'])
best_var_smoothing = 10.0 ** best_log10_vs

# --- 9. Final Naive Bayes Model Fitting (using Resampled Data) ---

nb_best_ = GaussianNB(var_smoothing=best_var_smoothing)
nb_best_.fit(X_train_res_nb, y_train_res_nb) # Fit on RESAMPLED data
y_pred_nb = nb_best_.predict(X_testNB) # Predict on encoded test data

print("\n--- Naive Bayes (Optimized) Results ---")
print(f"Best NB var_smoothing: {best_var_smoothing:.2e}")
print(f"Accuracy: {accuracy_score(y_testNB, y_pred_nb):.4f}")

Original X_train shape: (23431, 11)
Encoded X_train shape: (23431, 11)

Resampled X_train shape: (43579, 11)
Class distribution after ADASYN: Counter({1: 21983, 0: 21596})

--- Decision Tree (Optimized) Results ---
Best DT Params: {'criterion': 'entropy', 'max_depth': 4, 'min_samples_split': 3, 'min_samples_leaf': 2}
Accuracy: 0.8498

Top 5 Feature Importances:
nr.employed    0.386748
duration       0.328903
month          0.250605
euribor3m      0.024481
poutcome       0.005977
dtype: float64

--- Naive Bayes (Optimized) Results ---
Best NB var_smoothing: 2.07e-07
Accuracy: 0.8399
